##### setup

In [1]:
using TypedTables
using StatsBase
using SplitApplyCombine
using BenchmarkTools

In [2]:
using CovidSim_ilm

┌ Info: Precompiling CovidSim_ilm [top-level]
└ @ Base loading.jl:1317


In [3]:
cd(joinpath(homedir(), "Dropbox/Online Coursework/Covid/ilm-src"))

## Experiment with test data

In [4]:
tst = CovidSim_ilm.pop_data(100_000, age_dist=age_dist, intype=Int64, cols="track")

Table with 4 columns and 100000 rows:
      status  agegrp  cond  lag
    ┌──────────────────────────
 1  │ 1       1       0     0
 2  │ 1       1       0     0
 3  │ 1       1       0     0
 4  │ 1       1       0     0
 5  │ 1       1       0     0
 6  │ 1       1       0     0
 7  │ 1       1       0     0
 8  │ 1       1       0     0
 9  │ 1       1       0     0
 10 │ 1       1       0     0
 11 │ 1       1       0     0
 12 │ 1       1       0     0
 13 │ 1       1       0     0
 14 │ 1       1       0     0
 15 │ 1       1       0     0
 16 │ 1       1       0     0
 17 │ 1       1       0     0
 18 │ 1       1       0     0
 19 │ 1       1       0     0
 20 │ 1       1       0     0
 21 │ 1       1       0     0
 22 │ 1       1       0     0
 23 │ 1       1       0     0
 ⋮  │   ⋮       ⋮      ⋮     ⋮

In [5]:
tst.status[rand(1:100_000,5000)] .= infectious

5000-element view(::Vector{Int64}, [53532, 93484, 57466, 11365, 56618, 50581, 74929, 7084, 66513, 79252  …  7139, 77457, 26062, 27716, 80267, 64705, 81060, 60472, 64137, 76518]) with eltype Int64:
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2
 ⋮
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2
 2

In [6]:
countmap(tst.status)

Dict{Int64, Int64} with 2 entries:
  2 => 4886
  1 => 95114

In [39]:
@time in_idx = findall(tst.status .== 2)

  0.000143 seconds (8 allocations: 54.969 KiB)


4886-element Vector{Int64}:
     6
    27
   102
   181
   184
   191
   192
   211
   230
   233
   247
   289
   403
     ⋮
 99885
 99886
 99910
 99935
 99938
 99958
 99964
 99969
 99974
 99984
 99994
 99996

In [31]:
@time age3_sick_idx = findall((tst.agegrp .== 3) .& (tst.status .== 2))

  0.000278 seconds (12 allocations: 26.844 KiB)


1252-element Vector{Int64}:
 52201
 52244
 52245
 52284
 52286
 52312
 52355
 52368
 52414
 52432
 52436
 52440
 52459
     ⋮
 77481
 77487
 77564
 77567
 77602
 77622
 77623
 77642
 77645
 77653
 77656
 77695

In [16]:
 age3_idx = findall(tst.agegrp .== 3)

25500-element Vector{Int64}:
 52201
 52202
 52203
 52204
 52205
 52206
 52207
 52208
 52209
 52210
 52211
 52212
 52213
     ⋮
 77689
 77690
 77691
 77692
 77693
 77694
 77695
 77696
 77697
 77698
 77699
 77700

In [38]:
@btime begin
        age3_idx = findall($tst.agegrp .== 3)
        intersect($in_idx, age3_idx)
      end

  842.924 μs (25 allocations: 903.56 KiB)


1252-element Vector{Int64}:
 52201
 52244
 52245
 52284
 52286
 52312
 52355
 52368
 52414
 52432
 52436
 52440
 52459
     ⋮
 77481
 77487
 77564
 77567
 77602
 77622
 77623
 77642
 77645
 77653
 77656
 77695

In [41]:
@time intersect(in_idx, 1:100_000)

  0.002777 seconds (21 allocations: 2.359 MiB)


4886-element Vector{Int64}:
     6
    27
   102
   181
   184
   191
   192
   211
   230
   233
   247
   289
   403
     ⋮
 99885
 99886
 99910
 99935
 99938
 99958
 99964
 99969
 99974
 99984
 99994
 99996

In [46]:
@btime begin
    row = $tst[52224]
    row.lag == 4
end

  3.832 ns (0 allocations: 0 bytes)


false

In [49]:
@btime $tst.lag[52224] == 4

  1.398 ns (0 allocations: 0 bytes)


false

In [23]:
180 * .0008

0.14400000000000002

In [82]:
@btime begin
    leakage = 0.05
    rand() < leakage
end

  6.653 ns (0 allocations: 0 bytes)


true

In [156]:
@btime $tst.status[52520], $tst.agegrp[52520]

  1.885 ns (0 allocations: 0 bytes)


(2, 3)

In [193]:
@btime sickdict = Dict(i=>findall(($tst.agegrp .== i) .& ($tst.status .== 2)) for i in 1:5);

  471.311 μs (24 allocations: 122.39 KiB)


In [210]:
@btime $sickdict[1][50]

  5.462 ns (0 allocations: 0 bytes)


828

In [234]:
foo = @elapsed for i in eachindex(tst.status)
    if tst.status[i] == 2
        tst.lag[i] = 1
    end
end

0.015183456

In [236]:
bar = @elapsed begin
    infect_idx = CovidSim_ilm.optfindall(==(2), tst.status, 0.5)
    for p in infect_idx
        tst.lag[p] = 2
    end
end

0.000742404

### sample vs rand

In [242]:
@btime sample($age3_sick_idx, 5, replace=true)

  86.480 ns (1 allocation: 128 bytes)


5-element Vector{Int64}:
 67937
 59100
 75141
 55441
 56257

In [246]:
n = length(age3_sick_idx)
@btime begin
            who = rand(1:n,5)
            age3_sick_idx[who]
        end

  154.631 ns (3 allocations: 288 bytes)


5-element Vector{Int64}:
 75025
 53087
 61416
 75783
 71517

### We like sample